# Lab 00 · Arrume a casa antes de modelar
**Machine Learning na Prática · Dia 1 · Módulo 01: Dados & Datasets**

Neste notebook você vai fazer o que todo projeto de ML faz **antes** de treinar qualquer modelo.
Aqui **não há modelo nem métricas**: o objetivo é deixar os dados prontos. O treino começa no Lab 01, já com estes dados limpos.

| Etapa | O que fazer |
|---|---|
| 1. Inspecionar | tipos, estatísticas, faltantes e categorias |
| 2. Limpar | duplicados, categorias, outliers, alvo faltante |
| 3. Olhar | histogramas, dispersão e correlação |
| 4. Separar | features × alvo, one-hot e treino/teste |

Procure as células marcadas com **`# TODO`** e substitua os `___` pelo código certo.
Rode as células em ordem (`Shift + Enter`).

> **Gabarito.** Versão do professor, com todas as células resolvidas. Não compartilhe antes da atividade.

### Cola rápida de pandas
| Para… | use |
|---|---|
| ler um CSV | `pd.read_csv('arquivo.csv')` |
| tipos e faltantes de cada coluna | `df.info()` |
| estatísticas das colunas numéricas | `df.describe()` |
| faltantes por coluna | `df.isna().sum()` |
| contar valores de uma coluna | `df['col'].value_counts()` |
| remover duplicados | `df.drop_duplicates()` |
| limpar texto | `df['col'].str.strip().str.lower()` |
| remover linhas com faltante numa coluna | `df.dropna(subset=['col'])` |
| mapa de correlação | `sns.heatmap(df.corr(numeric_only=True), annot=True)` |

### Bibliotecas
No **Google Colab** tudo já vem instalado: esta célula só confere. Em outro ambiente (Jupyter no seu PC), ela instala o que faltar.

In [ ]:
import importlib.util, subprocess, sys
pacotes = {'numpy': 'numpy', 'pandas': 'pandas', 'matplotlib': 'matplotlib', 'seaborn': 'seaborn', 'sklearn': 'scikit-learn'}
faltando = [pip for modulo, pip in pacotes.items() if importlib.util.find_spec(modulo) is None]
if faltando:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *faltando])
print('Bibliotecas prontas!' if not faltando else f'Instalado agora: {", ".join(faltando)}')

## 0 · Preparação

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
print('Tudo pronto!')

### O dataset: desempenho de estudantes
Os dados abaixo são **sintéticos** (gerados aqui mesmo, para não depender de download), mas imitam problemas reais:
alguém digitou errado, algumas linhas foram copiadas duas vezes, o turno foi escrito de jeitos diferentes…

Rode a célula para criar o arquivo `alunos.csv`. Você **não precisa** entender o código de geração.

In [ ]:
def gerar_dados(n=400, seed=42):
    rng = np.random.default_rng(seed)
    horas = np.clip(rng.normal(6, 2.2, n), 0.5, 14).round(1)
    faltas = rng.poisson(3, n)
    turno = rng.choice(['manhã', 'tarde', 'noite'], n, p=[.45, .3, .25])
    monitoria = rng.choice(['sim', 'não'], n, p=[.35, .65])
    idade = rng.integers(17, 31, n)
    renda = np.round(rng.lognormal(8.2, 0.5, n), -1)          # R$ por mês, escala grande
    nota = (2.0 + 0.45 * horas - 0.25 * faltas + 0.6 * (monitoria == 'sim')
            - 0.4 * (turno == 'noite') + rng.normal(0, 0.9, n))
    nota = np.clip(nota, 0, 10).round(1)
    df = pd.DataFrame({'horas_estudo': horas, 'faltas': faltas, 'turno': turno,
                       'monitoria': monitoria, 'idade': idade, 'renda_familiar': renda,
                       'nota_final': nota})
    df['situacao'] = np.where(df['nota_final'] >= 6, 'aprovado', 'reprovado')

    # ---- problemas do mundo real ----
    sujo = rng.choice(n, 40, replace=False)
    df.loc[sujo[:14], 'turno'] = rng.choice(['Manhã', 'MANHÃ ', 'manha', 'Noite', ' tarde'], 14)
    df.loc[rng.choice(n, 20, replace=False), 'horas_estudo'] = np.nan
    df.loc[rng.choice(n, 15, replace=False), 'faltas'] = np.nan
    df.loc[rng.choice(n, 8, replace=False), 'nota_final'] = np.nan
    outl = rng.choice(n, 4, replace=False)
    df.loc[outl, 'horas_estudo'] = df.loc[outl, 'horas_estudo'].fillna(7) * 10   # vírgula esquecida
    df = pd.concat([df, df.sample(12, random_state=seed)], ignore_index=True)   # duplicados
    return df.sample(frac=1, random_state=seed).reset_index(drop=True)

gerar_dados().to_csv('alunos.csv', index=False)
print('alunos.csv criado!')

## 1 · Inspecionar

In [ ]:
# TODO: leia o arquivo 'alunos.csv' com o pandas
df = pd.read_csv('alunos.csv')
df.head()

> 🔎 **Gabarito · o que observar**
>
> - Cada linha é um aluno; `nota_final` é o **alvo** e as outras colunas são candidatas a feature.
> - `renda_familiar` está na casa dos milhares e `horas_estudo` na casa das unidades: escalas muito diferentes (isso volta na padronização).
> - `situacao` parece só mais uma coluna, mas ela é **derivada da nota** (veja a seção 4.1).

In [ ]:
# TODO: tipos de cada coluna e quantos valores não nulos ela tem
df.info()

> 🔎 **Gabarito · o que observar**
>
> - **412 linhas × 8 colunas**, mas o gerador criou 400 alunos: as 12 a mais são duplicados (seção 2).
> - O `Non-Null Count` já denuncia os faltantes: `horas_estudo` (393), `faltas` (395) e `nota_final` (404) têm menos de 412.
> - `faltas` é contagem (inteiro), mas aparece como `float64`: no pandas, basta **um `NaN`** para a coluna inteira virar float.
> - `turno`, `monitoria` e `situacao` são texto (`object` ou `str`, conforme a versão do pandas) e vão precisar virar números.

In [ ]:
# TODO: estatísticas das colunas numéricas. Repare no máximo de horas_estudo!
df.describe()

> 🔎 **Gabarito · o que observar**
>
> - **`horas_estudo`: máximo 70**, enquanto 75% dos alunos estudam até 7,4 h. É o outlier da “vírgula esquecida”.
> - Esse outlier contamina as estatísticas: a média (6,47) já passa da mediana (6,0) e o desvio (5,76) está inflado. Depois da limpeza, o desvio cai para cerca de 2,2.
> - `nota_final` vai de 0 a 8, com média ≈ 4,1: **a maioria reprova** (só 30 dos 392 alunos finais têm nota ≥ 6).
> - `renda_familiar` tem máximo 17 060 contra mediana 3 720. É cauda longa, mas **plausível**: não é erro de digitação e não deve ser apagada.
> - `idade` (17 a 30) e `faltas` (0 a 9) estão dentro do esperado.

In [ ]:
# TODO: quantos valores faltantes há em cada coluna?
df.isna().sum()

> 🔎 **Gabarito · o que observar**
>
> - Faltantes: `horas_estudo` 19, `faltas` 17, `nota_final` 8.
> - Faltante em **feature** dá para preencher (imputar); faltante no **alvo** não, porque não há o que ensinar ao modelo com essa linha (seção 2).

In [ ]:
# TODO: quais valores aparecem na coluna 'turno'?
df['turno'].value_counts()

> 🔎 **Gabarito · o que observar**
>
> - Deveriam ser 3 categorias, mas aparecem **8**: `manha`, `Manhã`, `MANHÃ ` (com espaço), `Noite`, ` tarde`…
> - Para o computador são valores diferentes. Sem limpar, o one-hot criaria 8 colunas para 3 turnos de verdade.

**Para pensar:** liste os problemas que você encontrou. Quais colunas têm faltantes? O que há de estranho em `turno` e em `horas_estudo`?

*Sua resposta:* …

> 🔎 **Gabarito · o que observar**
>
> **Resposta esperada:** faltantes em `horas_estudo`, `faltas` e `nota_final`; `turno` escrito de vários jeitos;
> `horas_estudo` com valores absurdos (70, 68, 64); 12 linhas duplicadas; e escalas muito diferentes entre as colunas numéricas.

## 2 · Limpar

In [ ]:
# TODO: conte as linhas duplicadas e remova-as
print('duplicados:', df.duplicated().sum())
df = df.drop_duplicates()
print('linhas agora:', len(df))

> 🔎 **Gabarito · o que observar**
>
> - **12 duplicados** → 400 linhas, exatamente o número de alunos gerados.
> - Duplicado é perigoso porque a mesma linha pode cair no treino **e** no teste: o modelo “vê” a resposta antes da prova.

Para o computador, `"manhã"`, `"Manhã"` e `"MANHÃ "` são três valores diferentes.
Padronize: tire os espaços das pontas, deixe tudo minúsculo e corrija `"manha"` sem acento.

In [ ]:
# TODO: padronize a coluna turno
df['turno'] = df['turno'].str.strip().str.lower()
df['turno'] = df['turno'].replace({'manha': 'manhã'})
df['turno'].value_counts()

> 🔎 **Gabarito · o que observar**
>
> - Agora são só 3 categorias: manhã 193, tarde 104, noite 103.
> - A ordem importa: primeiro `strip` e `lower`, depois o `replace` de `manha` → `manhã`.

Ninguém estuda 70 horas *extras* por semana: provavelmente alguém esqueceu a vírgula (7,0 → 70).
Como **não temos certeza** do valor certo, vamos apagar esses valores (virar `NaN`).

In [ ]:
# TODO: transforme em NaN os valores de horas_estudo acima de 24
df.loc[df['horas_estudo'] > 24, 'horas_estudo'] = np.nan
df['horas_estudo'].max()

> 🔎 **Gabarito · o que observar**
>
> - Os três valores absurdos (70, 68 e 64) viram `NaN`.
> - Depois da regra, o máximo vira **15**.
> - **Pegadinha:** esse 15 também é um erro (o valor real era 1,5), mas 15 h por semana é plausível, então a regra `> 24` não o pega.
>   Nem todo erro é detectável por regra simples. Ele vai reaparecer no Lab 01, puxando a reta para baixo.

Uma linha sem `nota_final` não ensina nada: remova-a.
Os faltantes nas **features** (horas, faltas) **ficam**: no Lab 01, eles serão preenchidos usando só o treino.

In [ ]:
# TODO: remova as linhas sem nota_final
df = df.dropna(subset=['nota_final'])
print(len(df), 'linhas')
df.isna().sum()

> 🔎 **Gabarito · o que observar**
>
> - **392 linhas** ficam. `nota_final` não tem mais faltantes.
> - `horas_estudo` agora tem 21 faltantes (inclui os 3 outliers que viraram `NaN`) e `faltas` tem 15. Eles **continuam** faltando de propósito: quem preenche é o `Pipeline` do Lab 01, olhando só o treino.

## 3 · Olhar antes de modelar

In [ ]:
# TODO: histogramas de todas as colunas numéricas
df.hist(bins=20, figsize=(12, 6))
plt.tight_layout(); plt.show()

> 🔎 **Gabarito · o que observar**
>
> - `renda_familiar` é **assimétrica à direita** (cauda longa de rendas altas).
> - `horas_estudo` ficou aproximadamente simétrica depois da limpeza.
> - `faltas` é discreta (só inteiros); `idade` é quase uniforme entre 17 e 30.
> - `nota_final` se concentra entre 3 e 5, e poucas passam de 6.

In [ ]:
# TODO: gráfico de dispersão horas_estudo × nota_final
plt.scatter(df['horas_estudo'], df['nota_final'], alpha=.5)
plt.xlabel('horas de estudo'); plt.ylabel('nota final'); plt.show()

> 🔎 **Gabarito · o que observar**
>
> - Tendência clara de **subida**: mais horas, nota maior em média.
> - Mas há muita dispersão: alunos com as mesmas horas têm notas bem diferentes. Horas sozinhas não explicam tudo.
> - Repare num ponto solto lá à direita, em 15 h com nota baixa: é o erro que a regra não pegou.

In [ ]:
# TODO: correlação entre as colunas numéricas, desenhada como um mapa de calor
corr = df.corr(numeric_only=True).round(2)
sns.heatmap(corr, annot=True, cmap='coolwarm', vmin=-1, vmax=1)
plt.show()

> 🔎 **Gabarito · o que observar**
>
> - `horas_estudo` × nota: **+0,62** (a mais forte). `faltas` × nota: **−0,31** (mais faltas, nota menor).
> - `idade` (0,03) e `renda_familiar` (−0,07) ≈ 0: praticamente **nenhuma relação linear** com a nota.
> - As features quase não se correlacionam entre si, o que é bom para a regressão.
> - A correlação só mede relação **linear** entre colunas numéricas: `turno` e `monitoria` nem aparecem no mapa.

**Para pensar:** quais features parecem ajudar a prever a nota? Alguma parece não ter relação nenhuma?
Lembre: correlação não é causalidade.

*Sua resposta:* …

> 🔎 **Gabarito · o que observar**
>
> **Resposta esperada:** `horas_estudo` (a mais forte) e `faltas` ajudam; `idade` e `renda_familiar` parecem não ter relação.
> `turno` e `monitoria` não aparecem no mapa porque são texto: depois do one-hot, o modelo do Lab 01 mostra se elas pesam.
> Correlação não é causa: a monitoria pode atrair justamente os alunos mais dedicados, por exemplo.

## 4 · Separar
### 4.1 Features × alvo — e a coluna que “conta a resposta”
O alvo é `nota_final`. A coluna `situacao` (aprovado/reprovado) é **calculada a partir da nota**: ela só existe *depois* que a nota sai.
Usá-la como feature seria **vazamento de dados** — o modelo “colaria” a resposta.

> 🔎 **Gabarito · o que observar**
>
> - Aprovado = nota ≥ 6; reprovado = nota abaixo de 6. A `situacao` separa as notas perfeitamente, sem sobreposição.
> - Com `situacao` como feature, o modelo descobriria a regra “nota ≥ 6” e pareceria ótimo, mas na vida real essa coluna **só existe depois** da nota. É vazamento.

In [ ]:
alvo = 'nota_final'
# TODO: as features são todas as colunas, menos o alvo e a coluna que vaza
X = df.drop(columns=[alvo, 'situacao'])
y = df[alvo]
X.head()

> 🔎 **Gabarito · o que observar**
>
> - Sobram **6 features**: 4 numéricas e 2 categóricas.

### 4.2 Categorias viram números (one-hot)
Modelos fazem contas: `turno` e `monitoria` precisam virar colunas de 0 e 1.

In [ ]:
# TODO: uma coluna 0/1 para cada categoria de turno e de monitoria
X = pd.get_dummies(X, columns=['turno', 'monitoria'], dtype=int)
X.head()

### 4.3 Separar treino e teste
O teste é a **prova inédita**: fica guardado até o fim. 80% para treino, 20% para teste.

In [ ]:
# TODO: 80% treino, 20% teste, random_state=42
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_tr.shape, X_te.shape)

> 🔎 **Gabarito · o que observar**
>
> - **313 linhas de treino e 79 de teste** (80/20 de 392).
> - O `random_state=42` faz todo mundo da turma ter a mesma divisão, então os números batem entre os colegas.

### 4.4 Salvar os dados limpos
O Lab 01 começa exatamente daqui.

In [ ]:
df.to_csv('alunos_limpo.csv', index=False)
print(df.shape, '→ alunos_limpo.csv')

> 🔎 **Gabarito · o que observar**
>
> - `(392, 8)`: o `alunos_limpo.csv` é o ponto de partida do Lab 01.
> - Ele ainda tem `NaN` nas features e ainda tem `situacao`. Isso é de propósito: quem cuida disso é o `Pipeline` de cada modelo.

## Para discutir
1. Apagar os outliers (`NaN`) ou dividir por 10: qual decisão você tomaria? O que muda nos dados?
2. Que outra coluna, num problema real, poderia “contar a resposta” como a `situacao`?
3. Por que o teste precisa ficar guardado até o fim?

> 🔎 **Gabarito · o que observar**
>
> **Pontos para a discussão:**
> 1. Dividir por 10 recupera o valor, se o palpite estiver certo; virar `NaN` é mais conservador e deixa o imputador do Lab 01 preencher com a mediana. Note que o 15 escapou das duas abordagens.
> 2. Exemplos: “cancelou o plano” para prever churn, data de alta para prever tempo de internação, “nota de recuperação” para prever reprovação.
> 3. Porque ele faz o papel de dados novos: se o teste ajudar em qualquer decisão (preparo, escolha de modelo), a nota final fica otimista demais.